# Compare normative models

In [1]:
import logging
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import seaborn as sns

import pcntoolkit.util.output
from pcntoolkit import (
    HBR,
    BsplineBasisFunction,
    NormalLikelihood,
    NormativeModel,
    NormData,
    load_fcon1000,
    make_prior,
)
from pcntoolkit.util.model_comparison import compare_hbr_models

sns.set_style("darkgrid")

# Suppress some annoying warnings and logs
pymc_logger = logging.getLogger("pymc")

pymc_logger.setLevel(logging.WARNING)
pymc_logger.propagate = False

warnings.simplefilter(action="ignore", category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'
pcntoolkit.util.output.Output.set_show_messages(True)

In [2]:
# Download an example dataset
norm_data: NormData = load_fcon1000()

# Select only a few features
features_to_model = [
    "WM-hypointensities",
    "Right-Lateral-Ventricle",
    # "Right-Amygdala",
    # "CortexVol",
]
norm_data = norm_data.sel({"response_vars": features_to_model})

# Split into train and test sets
train, test = norm_data.train_test_split()

Process: 3265 - 2026-05-22 16:15:49 - Removed 0 NANs
Process: 3265 - 2026-05-22 16:15:49 - Dataset "fcon1000" created.
    - 1078 observations
    - 1078 unique subjects
    - 1 covariates
    - 217 response variables
    - 2 batch effects:
    	sex (2)
	site (23)
    


In [3]:
mu1 = make_prior(
    # Mu is linear because we want to allow the mean to vary as a function of the covariates.
    linear=True,
    # The slope coefficients are assumed to be normally distributed, with a mean of 0 and a standard deviation of 10.
    slope=make_prior(dist_name="Normal", dist_params=(0.0, 5.0)),
    # The intercept is not random, because we want to compare to a model with random intercept
    intercept=make_prior(
        dist_name="Normal",
        dist_params=(0.0, 2.0),
    ),
    # We use a B-spline basis function to allow for non-linearity in the mean.
    basis_function=BsplineBasisFunction(basis_column=0, nknots=5, degree=3),
)
sigma1 = make_prior(
    # Sigma is also linear, because we want to allow the standard deviation to vary as a function of the covariates: heteroskedasticity.
    linear=True,
    # The slope coefficients are assumed to be normally distributed, with a mean of 0 and a standard deviation of 2.
    slope=make_prior(dist_name="Normal", dist_params=(0.0, 2.0)),
    # The intercept is not random, because we assume the intercept of the variance to be the same for all sites and sexes.
    intercept=make_prior(dist_name="Normal", dist_params=(1.0, 1.0)),
    # We use a B-spline basis function to allow for non-linearity in the standard deviation.
    basis_function=BsplineBasisFunction(basis_column=0, nknots=5, degree=3),
    # We use a softplus mapping to ensure that sigma is strictly positive.
    mapping="softplus",
    # We scale the softplus mapping by a factor of 3, to avoid spikes in the resulting density.
    # The parameters (a, b, c) provided to a mapping f are used as: f_abc(x) = f((x - a) / b) * b + c
    # This basically provides an affine transformation of the softplus function.
    # a -> horizontal shift
    # b -> scaling
    # c -> vertical shift
    # You can leave c out, and it will default to 0.
    mapping_params=(0.0, 3.0),
)
# Set the likelihood with the priors we just created.
likelihood1 = NormalLikelihood(mu1, sigma1)

template_hbr_1 = HBR(
    name="template",
    # The number of cores to use for sampling.
    cores=16,
    # Whether to show a progress bar during the model fitting.
    progressbar=True,
    # The number of draws to sample from the posterior per chain.
    draws=1500,
    # The number of tuning steps to run.
    tune=500,
    # The number of MCMC chains to run.
    chains=4,
    # The sampler to use for the model.
    nuts_sampler="nutpie",
    # The likelihood function to use for the model.
    likelihood=likelihood1,
)
model1 = NormativeModel(
    # The regression model to use for the normative model.
    template_regression_model=template_hbr_1,
    # Whether to save the model after fitting.
    savemodel=True,
    # Whether to evaluate the model after fitting.
    evaluate_model=True,
    # Whether to save the results after evaluation.
    saveresults=True,
    # Whether to save the plots after fitting.
    saveplots=False,
    # The directory to save the model, results, and plots.
    save_dir="resources/compare_hbr/model1",
    # The scaler to use for the input data. Can be either one of "standardize", "minmax", "robminmax", "none"
    inscaler="standardize",
    # The scaler to use for the output data. Can be either one of "standardize", "minmax", "robminmax", "none"
    outscaler="standardize",
)

In [4]:
mu2 = make_prior(
    # Mu is linear because we want to allow the mean to vary as a function of the covariates.
    linear=True,
    # The slope coefficients are assumed to be normally distributed, with a mean of 0 and a standard deviation of 10.
    slope=make_prior(dist_name="Normal", dist_params=(0.0, 5.0)),
    # The intercept is random, because we expect the intercept to vary between sites and sexes.
    intercept=make_prior(
        random=True,
        # Mu is the mean of the intercept, which is normally distributed with a mean of 0 and a standard deviation of 1.
        mu=make_prior(dist_name="Normal", dist_params=(0.0, 2.0)),
        # Sigma is the scale at which the intercepts vary. It is a positive parameter, so we have to map it to the positive domain.
        sigma=make_prior(dist_name="Normal", dist_params=(1.0, 0.5), mapping="softplus", mapping_params=(0.0, 2.0)),
    ),
    # We use a B-spline basis function to allow for non-linearity in the mean.
    basis_function=BsplineBasisFunction(basis_column=0, nknots=5, degree=3),
)
sigma2 = make_prior(
    # Sigma is also linear, because we want to allow the standard deviation to vary as a function of the covariates: heteroskedasticity.
    linear=True,
    # The slope coefficients are assumed to be normally distributed, with a mean of 0 and a standard deviation of 2.
    slope=make_prior(dist_name="Normal", dist_params=(0.0, 2.0)),
    # The intercept is not random, because we assume the intercept of the variance to be the same for all sites and sexes.
    intercept=make_prior(dist_name="Normal", dist_params=(1.0, 1.0)),
    # We use a B-spline basis function to allow for non-linearity in the standard deviation.
    basis_function=BsplineBasisFunction(basis_column=0, nknots=5, degree=3),
    # We use a softplus mapping to ensure that sigma is strictly positive.
    mapping="softplus",
    # We scale the softplus mapping by a factor of 3, to avoid spikes in the resulting density.
    # The parameters (a, b, c) provided to a mapping f are used as: f_abc(x) = f((x - a) / b) * b + c
    # This basically provides an affine transformation of the softplus function.
    # a -> horizontal shift
    # b -> scaling
    # c -> vertical shift
    # You can leave c out, and it will default to 0.
    mapping_params=(0.0, 3.0),
)
# Set the likelihood with the priors we just created.
likelihood2 = NormalLikelihood(mu2, sigma2)

template_hbr_2 = HBR(
    name="template",
    # The number of cores to use for sampling.
    cores=16,
    # Whether to show a progress bar during the model fitting.
    progressbar=True,
    # The number of draws to sample from the posterior per chain.
    draws=1500,
    # The number of tuning steps to run.
    tune=500,
    # The number of MCMC chains to run.
    chains=4,
    # The sampler to use for the model.
    nuts_sampler="nutpie",
    # The likelihood function to use for the model.
    likelihood=likelihood2,
)
model2 = NormativeModel(
    # The regression model to use for the normative model.
    template_regression_model=template_hbr_2,
    # Whether to save the model after fitting.
    savemodel=True,
    # Whether to evaluate the model after fitting.
    evaluate_model=True,
    # Whether to save the results after evaluation.
    saveresults=True,
    # Whether to save the plots after fitting.
    saveplots=False,
    # The directory to save the model, results, and plots.
    save_dir="resources/compare_hbr/model2",
    # The scaler to use for the input data. Can be either one of "standardize", "minmax", "robminmax", "none"
    inscaler="standardize",
    # The scaler to use for the output data. Can be either one of "standardize", "minmax", "robminmax", "none"
    outscaler="standardize",
)

In [5]:
model1.fit_predict(train, test)
model2.fit_predict(train, test)

Process: 3265 - 2026-05-22 16:15:49 - Fitting models on 2 response variables.
Process: 3265 - 2026-05-22 16:15:49 - Fitting model for WM-hypointensities.


/opt/hostedtoolcache/Python/3.12.13/x64/lib/python3.12/site-packages/pytensor/link/c/cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.11,127
,2000,0,0.11,63
,2000,0,0.11,127
,2000,0,0.11,95


Process: 3265 - 2026-05-22 16:16:24 - Fitting model for Right-Lateral-Ventricle.


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.12,63
,2000,0,0.15,63
,2000,0,0.14,31
,2000,0,0.13,31


...

Process: 3265 - 2026-05-22 16:18:14 - Computing centiles for Right-Lateral-Ventricle.


Process: 3265 - 2026-05-22 16:18:16 - Computing log-probabilities for 2 response variables.
Process: 3265 - 2026-05-22 16:18:16 - Computing log-probabilities for 2 response variables.
Process: 3265 - 2026-05-22 16:18:16 - Computing log-probabilities for WM-hypointensities.


Process: 3265 - 2026-05-22 16:18:17 - Computing log-probabilities for Right-Lateral-Ventricle.


Process: 3265 - 2026-05-22 16:18:18 - Computing yhat for 2 response variables.


<xarray.NormData> Size: 70kB
Dimensions:            (observations: 216, response_vars: 2, covariates: 1,
                        batch_effect_dims: 2, statistic: 13, centile: 5)
Coordinates:
  * observations       (observations) int64 2kB 756 769 692 616 ... 751 470 1043
  * response_vars      (response_vars) <U23 184B 'WM-hypointensities' 'Right-...
  * covariates         (covariates) <U3 12B 'age'
  * batch_effect_dims  (batch_effect_dims) <U4 32B 'sex' 'site'
  * statistic          (statistic) <U8 416B 'EXPV' 'Kurtosis' ... 'Skewness'
  * centile            (centile) float64 40B 0.05 0.25 0.5 0.75 0.95
Data variables:
    subject_ids        (observations) object 2kB 'Munchen_sub96752' ... 'Quee...
    Y                  (observations, response_vars) float64 3kB 2.721e+03 .....
    X                  (observations, covariates) float64 2kB 63.0 ... 23.0
    batch_effects      (observations, batch_effect_dims) <U17 29kB 'F' ... 'Q...
    Z                  (observations, response_vars) float64 3kB 0.5254 ... 1...
    baseline_logp      (observations, response_vars) float64 3kB -3.66 ... -1...
    logp               (observations, response_vars) float64 3kB -1.705 ... -...
    Yhat               (observations, response_vars) float64 3kB 1.945e+03 .....
    statistics         (response_vars, statistic) float64 208B 0.3624 ... 1.446
    centiles           (centile, observations, response_vars) float64 17kB -5...
Attributes:
    real_ids:                       True
    is_scaled:                      False
    name:                           fcon1000_test
    unique_batch_effects:           {np.str_('sex'): ['M', 'F'], np.str_('sit...
    batch_effect_counts:            defaultdict(<function NormData.register_b...
    covariate_ranges:               {np.str_('age'): {'min': 7.88, 'max': 85.0}}
    batch_effect_covariate_ranges:  {np.str_('sex'): {'M': {np.str_('age'): {...

In [6]:
# Delete references to model objects to ensure what follows will work for models saved to disk too
del model1
del model2

In [7]:
dct = {"model1": "resources/compare_hbr/model1", "model2": "resources/compare_hbr/model2"}
comparison = compare_hbr_models(dct)

Process: 3265 - 2026-05-22 16:18:19 - Dataset "synthesized" created.
    - 92 observations
    - 92 unique subjects
    - 1 covariates
    - 2 response variables
    - 2 batch effects:
    	sex (2)
	site (21)
    
Process: 3265 - 2026-05-22 16:18:19 - Synthesizing data for 2 response variables.
Process: 3265 - 2026-05-22 16:18:19 - Synthesizing data for WM-hypointensities.


Process: 3265 - 2026-05-22 16:18:20 - Synthesizing data for Right-Lateral-Ventricle.


Process: 3265 - 2026-05-22 16:18:20 - Making predictions on 2 response variables.
Process: 3265 - 2026-05-22 16:18:20 - Computing z-scores for 2 response variables.
Process: 3265 - 2026-05-22 16:18:20 - Computing z-scores for WM-hypointensities.


Process: 3265 - 2026-05-22 16:18:20 - Computing z-scores for Right-Lateral-Ventricle.


Process: 3265 - 2026-05-22 16:18:21 - Computing centiles for 2 response variables.
Process: 3265 - 2026-05-22 16:18:21 - Computing centiles for WM-hypointensities.


...

Output()

/opt/hostedtoolcache/Python/3.12.13/x64/lib/python3.12/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/opt/hostedtoolcache/Python/3.12.13/x64/lib/python3.12/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


Output()

Output()

/opt/hostedtoolcache/Python/3.12.13/x64/lib/python3.12/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/opt/hostedtoolcache/Python/3.12.13/x64/lib/python3.12/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


In [8]:
for k, v in comparison.items():
    print(k)
    display(v)

WM-hypointensities


,rank,elpd_loo,p_loo,elpd_diff,weight,se,dse,warning,scale
model2,0,-144.900425,8.252381,0.000000,0.538644,10.247952,0.000000,True,log
model1,1,-157.206130,6.991130,12.305704,0.461356,11.433660,15.248923,True,log


Right-Lateral-Ventricle


,rank,elpd_loo,p_loo,elpd_diff,weight,se,dse,warning,scale
model1,0,-146.825669,3.835875,0.00000,0.578616,7.063790,0.000000,True,log
model2,1,-166.488959,16.411194,19.66329,0.421384,12.459351,15.072901,True,log
